# CBP CFP Core Feasibility Pilot — Real Faces

Notebook này chuyên cho **Kaggle CFPW/CFP raw dataset** có cấu trúc:

```text
cfp-dataset/
├── Data/
│   ├── Fiducials/
│   └── Images/
├── list_name.txt
├── Protocol/
└── Readme.txt
```

Mục tiêu pilot chỉ tập trung vào 3 câu hỏi lõi:

1. **Gate A / E1 — phenomenon:** các states gần giống nhau về mean behavior \(g,r\) có còn khác evidence/influence trên unseen identities không?
2. **Gate B / E2 — decision relevance:** evidence-aware CBP có tạo genuine-yield tốt hơn/không kém mean-only compression tại matched FMR/coverage không?
3. **E3 — router fidelity:** inductive single-image router có giữ teacher policy behavior trên unseen identities không?

### Frozen FR model
Dùng **InsightFace `buffalo_l`**, backbone frozen hoàn toàn.

> Đây là **mechanism pilot** trên dữ liệu CFP, không phải official CFP-FP pair-protocol benchmark score.


In [ ]:
# ============================================================
# 0. USER CONFIG — thường chỉ cần sửa 2 path đầu
# ============================================================

USE_GOOGLE_DRIVE = True

# Folder project trên Google Drive
PROJECT_DIR_DRIVE = "/content/drive/MyDrive/CBP/CBP_full_extract"

# File ZIP CFP tải từ Kaggle và upload nguyên file lên Drive.
CFP_ZIP_DRIVE = "/content/drive/MyDrive/datasets/CFP/cfp-dataset.zip"

# Colab sẽ tự giải nén ZIP vào local runtime để đọc ảnh nhanh hơn Drive.
CFP_LOCAL_DIR = "/content/CFP"

DATASET_NAME = "CFP_core_pilot"

# Frozen InsightFace model pack
MODEL_NAME = "buffalo_l"

# Cache/output trên Drive
EMBEDDINGS_NPZ = "/content/drive/MyDrive/CBP/cache/cfp_buffalo_l_embeddings.npz"
RUN_OUTPUT_DIR = "/content/drive/MyDrive/CBP/runs/cfp_buffalo_l_core_pilot"

# True chỉ để smoke test nhanh, không dùng làm research evidence.
FAST_MODE = False

# Chạy pytest + E0 trước real-face pilot.
RUN_CODE_SANITY = True

# True nếu muốn extract embedding lại từ đầu.
FORCE_REEXTRACT = False

SEED = 20261021


In [ ]:
# ============================================================
# 1. Mount Drive + copy source về local Colab disk
# ============================================================

import os, sys, shutil, subprocess, json, time
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

src = Path(PROJECT_DIR_DRIVE)
assert (src / "pyproject.toml").exists(), f"Không thấy project tại: {src}"

PROJECT_DIR = Path("/content/CBP_full_extract")
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(src, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print("PROJECT_DIR =", PROJECT_DIR)
print("Python =", sys.version)

gpu = subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || true"],
    capture_output=True, text=True
).stdout.strip()
print("GPU =", gpu if gpu else "No GPU detected")


In [ ]:
# ============================================================
# 2. Install dependencies
# ============================================================

import subprocess, sys, os

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "insightface>=0.7.3", "onnxruntime-gpu>=1.16"
])

import torch
import onnxruntime as ort

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("onnxruntime providers:", ort.get_available_providers())

PROVIDER = (
    "CUDAExecutionProvider"
    if "CUDAExecutionProvider" in ort.get_available_providers()
    else "CPUExecutionProvider"
)
print("InsightFace provider:", PROVIDER)


## 3. CFP dataset preflight

Bạn chỉ cần upload **nguyên file ZIP** từ Kaggle lên Google Drive.

Notebook sẽ:

1. kiểm tra file `CFP_ZIP_DRIVE`;
2. giải nén ZIP vào `/content/CFP` trên Colab;
3. tự tìm `cfp-dataset/Data/Images`;
4. build identity manifest;
5. kiểm tra số identities/captures trước khi chạy InsightFace.

Cách này nhanh hơn nhiều so với upload 7,000 file ảnh riêng lẻ lên Drive.


In [ ]:
# ============================================================
# 3. Unzip CFP locally + structure check + manifest
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import subprocess, sys, os, json, shutil, zipfile

zip_path = Path(CFP_ZIP_DRIVE)
assert zip_path.exists(), f"Không thấy CFP ZIP trên Drive: {zip_path}"

local_root = Path(CFP_LOCAL_DIR)

# Giải nén vào local Colab disk để đọc nhanh hơn Google Drive.
# Nếu đã giải nén trong runtime hiện tại thì reuse.
if not local_root.exists() or not any(local_root.iterdir()):
    local_root.mkdir(parents=True, exist_ok=True)
    print("Extracting CFP ZIP to local runtime...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(local_root)
else:
    print("Reuse existing local CFP extraction:", local_root)

# Auto-detect mirror structure.
candidates = [
    local_root / "cfp-dataset" / "Data" / "Images",
    local_root / "Data" / "Images",
]

# Một số ZIP có thêm một top-level folder.
candidates += list(local_root.glob("*/cfp-dataset/Data/Images"))
candidates += list(local_root.glob("*/Data/Images"))

image_root = next((p for p in candidates if p.exists()), None)

if image_root is None:
    print("Không tìm thấy Data/Images. Các mục cấp 1 sau khi giải nén:")
    for p in sorted(local_root.iterdir()):
        print(" -", p.name)
    raise FileNotFoundError(
        "Không tìm thấy cấu trúc CFP raw */Data/Images sau khi giải nén ZIP."
    )

# Suy ra dataset root để kiểm tra Protocol/Readme.
cfp_root = image_root.parent.parent

print("ZIP           :", zip_path)
print("Local root    :", local_root)
print("CFP root      :", cfp_root)
print("Image root    :", image_root)
print("Protocol found:", (cfp_root / "Protocol").exists())
print("Readme found  :", (cfp_root / "Readme.txt").exists())

identity_dirs = sorted([p for p in image_root.iterdir() if p.is_dir()])
print("Identity folders detected:", len(identity_dirs))
print("First identities:", [p.name for p in identity_dirs[:10]])

assert len(identity_dirs) > 0, f"Không thấy identity folders trong {image_root}"

data_dir = PROJECT_DIR / "data"
data_dir.mkdir(exist_ok=True)
manifest_path = data_dir / f"{DATASET_NAME}_manifest.csv"

subprocess.check_call([
    sys.executable, "-m", "scripts.build_manifest",
    "--root", str(image_root),
    "--output", str(manifest_path),
], cwd=PROJECT_DIR)

df = pd.read_csv(manifest_path)
required = {"image_path", "identity_id"}
assert required.issubset(df.columns), f"Manifest thiếu cột: {required - set(df.columns)}"

counts = df.groupby("identity_id").size().sort_values()
summary = {
    "images": int(len(df)),
    "identities": int(df["identity_id"].nunique()),
    "identities_ge_2_captures": int((counts >= 2).sum()),
    "median_captures_per_identity": float(counts.median()),
    "min_captures": int(counts.min()),
    "max_captures": int(counts.max()),
}
print(json.dumps(summary, indent=2))

# CFP raw chuẩn kỳ vọng xấp xỉ 500 identities / 7000 images.
if summary["images"] < 6000 or summary["identities"] < 400:
    print("\nWARNING: số ảnh/identity thấp hơn kỳ vọng; kiểm tra lại ZIP/cấu trúc.")
else:
    print("\nCFP dataset size nhìn hợp lý cho core pilot.")

display(counts.describe().to_frame("captures_per_identity"))


In [ ]:
# ============================================================
# 4. Frozen InsightFace embedding extraction
# ============================================================

from pathlib import Path
import subprocess, sys, json

emb_path = Path(EMBEDDINGS_NPZ)
emb_path.parent.mkdir(parents=True, exist_ok=True)

if emb_path.exists() and not FORCE_REEXTRACT:
    print("Reuse embedding cache:", emb_path)
else:
    cmd = [
        sys.executable, "-m", "scripts.extract_embeddings",
        "--manifest", str(manifest_path),
        "--output", str(emb_path),
        "--model", MODEL_NAME,
        "--provider", PROVIDER,
        "--ctx-id", "0" if PROVIDER == "CUDAExecutionProvider" else "-1",
        "--metadata", str(emb_path) + ".metadata.json",
        "--failure-log", str(emb_path) + ".failures.tsv",
    ]
    print("Running embedding extraction...")
    subprocess.check_call(cmd, cwd=PROJECT_DIR)

print("Embeddings:", emb_path)
print("Size MB:", round(emb_path.stat().st_size / 1024**2, 2))

meta_path = Path(str(emb_path) + ".metadata.json")
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print(json.dumps(meta, indent=2)[:6000])


In [ ]:
# ============================================================
# 5. Optional code/math sanity
# ============================================================

import subprocess, sys, json
from pathlib import Path

if RUN_CODE_SANITY:
    print("pytest...")
    subprocess.check_call([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR)

    e0_out = PROJECT_DIR / "outputs" / "e0_colab" / "report.json"
    e0_out.parent.mkdir(parents=True, exist_ok=True)

    print("\nE0 exact-vs-Monte-Carlo...")
    subprocess.check_call([
        sys.executable, "-m", "scripts.run_e0_synthetic",
        "--states", "16",
        "--identities", "50",
        "--repetitions", "5000" if FAST_MODE else "20000",
        "--seed", str(SEED),
        "--output", str(e0_out),
    ], cwd=PROJECT_DIR)

    e0 = json.loads(e0_out.read_text(encoding="utf-8"))
    print("E0 relative variance error:", e0.get("relative_error"))
else:
    print("Skipped code/math sanity.")


## 6. Focused CBP pilot config

Để trả lời **core feasibility** trước, notebook giảm trọng số của phần paper-scale:

- vẫn giữ identity-disjoint split;
- vẫn giữ CBP vs mean-only / basic controls;
- vẫn train router và refit deployed moments;
- ưu tiên FMR \(10^{-2}\), giữ \(10^{-3}\) như secondary;
- continuous/full-image neural comparator được tắt trong core pilot để giảm compute.

Sau khi Gate A/B có signal mới bật full Gate C/D.

In [ ]:
# ============================================================
# 6. Create focused pilot config
# ============================================================

import yaml
from pathlib import Path

base_cfg = yaml.safe_load((PROJECT_DIR / "configs" / "pilot.yaml").read_text(encoding="utf-8"))

base_cfg["seed"] = SEED
base_cfg["embedding_npz"] = str(emb_path)
base_cfg["output_dir"] = RUN_OUTPUT_DIR

# Core feasibility: không cần full expensive challenge ngay.
base_cfg["run_continuous_baseline"] = False
base_cfg["run_image_neural_baseline"] = False
base_cfg["certify_candidate_family"] = False

# Primary core pilot
base_cfg["k_candidates"] = [4, 8, 16]
base_cfg["n_states"] = 64
base_cfg["spectral_dim"] = 8

if FAST_MODE:
    base_cfg["max_identities_per_split"] = min(base_cfg.get("max_identities_per_split") or 80, 80)
    base_cfg["router_epochs"] = 20
    base_cfg["router_patience"] = 5
    base_cfg["phenomenon_permutations"] = 100
    base_cfg["bootstrap_repetitions"] = 100
    print("FAST_MODE=True: smoke test only.")
else:
    base_cfg["phenomenon_permutations"] = 1000
    base_cfg["bootstrap_repetitions"] = 1000

pilot_cfg = PROJECT_DIR / "configs" / "colab_core_pilot.yaml"
pilot_cfg.write_text(yaml.safe_dump(base_cfg, sort_keys=False), encoding="utf-8")
print(pilot_cfg.read_text(encoding="utf-8"))


In [ ]:
# ============================================================
# 7. Run REAL-FACE core pilot
# ============================================================

import subprocess, sys, time

start = time.time()
subprocess.check_call([
    sys.executable, "-m", "scripts.run_full_experiment",
    "--config", str(pilot_cfg),
], cwd=PROJECT_DIR)

print(f"Finished in {(time.time()-start)/60:.1f} min")
print("Output:", RUN_OUTPUT_DIR)


In [ ]:
# ============================================================
# 8. Read only the core scientific evidence first
# ============================================================

from pathlib import Path
import json

report_path = Path(RUN_OUTPUT_DIR) / "report.json"
assert report_path.exists(), report_path
report = json.loads(report_path.read_text(encoding="utf-8"))

gates = report.get("gates", {})

print("=== GATE A: REAL-FACE PHENOMENON ===")
print(json.dumps(gates.get("A_phenomenon", {}), indent=2)[:15000])

print("\n=== GATE B: DECISION RELEVANCE ===")
print(json.dumps(gates.get("B_decision_relevance", {}), indent=2)[:15000])

print("\n=== SELECTED REPRESENTATION ===")
print(json.dumps(report.get("selection", {}), indent=2)[:10000])

# Tìm mọi router_transfer block trong report mà không phụ thuộc exact nesting.
router_blocks = []
def walk(x, path="root"):
    if isinstance(x, dict):
        for k, v in x.items():
            p = f"{path}.{k}"
            if k == "router_transfer":
                router_blocks.append((p, v))
            walk(v, p)
    elif isinstance(x, list):
        for i, v in enumerate(x):
            walk(v, f"{path}[{i}]")

walk(report)

print("\n=== ROUTER TRANSFER BLOCKS ===")
for path, block in router_blocks[:20]:
    print("\n", path)
    print(json.dumps(block, indent=2)[:5000])


In [ ]:
# ============================================================
# 9. Core feasibility verdict
# ============================================================

def passed(name):
    x = gates.get(name, {})
    return x.get("pass") if isinstance(x, dict) else None

A = passed("A_phenomenon")
B = passed("B_decision_relevance")

print("CBP CORE REAL-FACE PILOT")
print("=" * 50)
print("Gate A phenomenon       :", "PASS" if A else "FAIL/NOT ESTABLISHED")
print("Gate B decision relevance:", "PASS" if B else "FAIL/NOT ESTABLISHED")

if A is True and B is True:
    print("\nCORE FEASIBILITY SIGNAL: POSITIVE")
    print("Next: second frozen backbone + full dense/continuous challenge + larger dataset.")
elif A is True and B is not True:
    print("\nPhenomenon có signal nhưng chưa có downstream decision value.")
    print("Không nên tăng compute để ép thắng; inspect representation/policy.")
else:
    print("\nCore phenomenon chưa được establish trên pilot này.")
    print("Inspect dataset, state construction, controls và backbone trước khi mở rộng.")

print("\nGate C/D nếu có trong report chỉ xem exploratory ở notebook này.")


In [ ]:
# ============================================================
# 10. Export run bundle
# ============================================================

import shutil
from pathlib import Path

run_dir = Path(RUN_OUTPUT_DIR)
zip_base = run_dir.parent / (run_dir.name + "_export")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=run_dir)
print("Exported:", zip_path)
